# 05 — Benchmarking on a Known Network

**Companion to `docs/05_discovery_algorithms.md` & `docs/06_ci_tests.md`
(Phases 4–5).**

Because the synthetic network has a *known* Markov Blanket, we can measure
discovery quality directly with precision / recall / F1 and study how it depends
on sample size and α. Everything is self-contained.


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

## 0. Inline tooling: G-test, network generator, GS algorithm

In [2]:
def _table(x, y):
    xs, xi = np.unique(x, return_inverse=True)
    ys, yi = np.unique(y, return_inverse=True)
    t = np.zeros((len(xs), len(ys))); np.add.at(t, (xi, yi), 1); return t

def g_test(x, y, z=None):
    z = z or []; x = np.asarray(x); y = np.asarray(y)
    if not z:
        tabs = [_table(x, y)]
    else:
        zmat = np.column_stack([np.asarray(c).reshape(-1) for c in z])
        _, inv = np.unique(zmat, axis=0, return_inverse=True)
        tabs = [_table(x[inv==s], y[inv==s]) for s in range(inv.max()+1)]
    G, dof = 0.0, 0
    for tab in tabs:
        n = tab.sum()
        if n == 0: continue
        row = tab.sum(1, keepdims=True); col = tab.sum(0, keepdims=True)
        e = row @ col / n; nz = (tab > 0) & (e > 0)
        G += 2.0 * np.sum(tab[nz] * np.log(tab[nz] / e[nz]))
        dof += max(int(np.count_nonzero(row))-1,0) * max(int(np.count_nonzero(col))-1,0)
    return float(chi2.sf(G, max(dof,1)))      # return p-value only

def make_network(n=8000, seed=1):
    r = np.random.default_rng(seed)
    A = r.integers(0,2,n); B = r.integers(0,2,n); S = r.integers(0,2,n); D = r.integers(0,2,n)
    T = (r.random(n) < np.where(A==1,0.8,0.2)).astype(int)
    logit = -1.5 + 1.6*T + 1.6*B + 1.6*S
    C = (r.random(n) < 1/(1+np.exp(-logit))).astype(int)
    return pd.DataFrame({"A":A,"B":B,"S":S,"T":T,"C":C,"D":D}), {"A","C","B","S"}

def grow_shrink(df, target, alpha=0.05):
    cols = {k: df[k].to_numpy() for k in df.columns}
    variables = [v for v in df.columns if v != target]
    def indep(x, Sset): return g_test(cols[x], cols[target], [cols[v] for v in Sset]) > alpha
    def stat(x, Sset):  return g_test(cols[x], cols[target], [cols[v] for v in Sset])
    S = []
    changed = True
    while changed:
        changed = False
        cands = sorted([v for v in variables if v not in S], key=lambda x: stat(x, S))
        for x in cands:
            if not indep(x, S):
                S.append(x); changed = True; break
    changed = True
    while changed:
        changed = False
        for x in list(S):
            rest = [v for v in S if v != x]
            if indep(x, rest):
                S.remove(x); changed = True; break
    return S

def evaluate(found, true):
    found, true = set(found), set(true)
    tp, fp, fn = len(found & true), len(found - true), len(true - found)
    precision = tp/(tp+fp) if tp+fp else 1.0
    recall    = tp/(tp+fn) if tp+fn else 1.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return precision, recall, f1

## 1. Quality vs sample size

CI tests need data. With too few samples the test loses power and the algorithm
misses true blanket members (recall drops). Watch F1 climb toward 1.0.


In [3]:
print(f"{'N':>7} | {'prec':>5} {'rec':>5} {'f1':>5}")
print("-"*30)
for n in [200, 500, 1000, 4000, 12000]:
    df, true_mb = make_network(n=n, seed=1)
    p, r, f = evaluate(grow_shrink(df, "T"), true_mb)
    print(f"{n:>7} | {p:5.2f} {r:5.2f} {f:5.2f}")

      N |  prec   rec    f1
------------------------------
    200 |  1.00  0.50  0.67
    500 |  1.00  0.50  0.67
   1000 |  1.00  0.50  0.67
   4000 |  1.00  1.00  1.00


  12000 |  1.00  1.00  1.00


## 2. Quality vs α

α trades false positives against false negatives. The sweet spot is usually
0.01–0.05.


In [4]:
df, true_mb = make_network(n=8000, seed=1)
print(f"{'alpha':>7} | {'prec':>5} {'rec':>5} {'f1':>5} | blanket")
print("-"*52)
for a in [0.001, 0.005, 0.01, 0.05, 0.1, 0.3]:
    res = grow_shrink(df, "T", alpha=a)
    p, r, f = evaluate(res, true_mb)
    print(f"{a:>7} | {p:5.2f} {r:5.2f} {f:5.2f} | {sorted(res)}")

  alpha |  prec   rec    f1 | blanket
----------------------------------------------------
  0.001 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']


  0.005 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']
   0.01 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']


   0.05 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']
    0.1 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']


    0.3 |  1.00  1.00  1.00 | ['A', 'B', 'C', 'S']


## 3. Stability across random seeds

A trustworthy method gives consistent blankets across resamples. Report how
often it recovers the blanket exactly.


In [5]:
trials = 25
exact = 0
for s in range(trials):
    df, true_mb = make_network(n=6000, seed=s)
    exact += (set(grow_shrink(df, "T")) == true_mb)
print(f"exact recovery: {exact}/{trials} = {exact/trials:.0%}")

exact recovery: 23/25 = 92%


## 4. Degrees of freedom blow-up (why small conditioning sets matter)

Re-deriving the lesson from doc 06 quantitatively: conditioning two independent
variables on `k` extra binaries multiplies strata by `2^k`. With fixed N, cells
empty and the p-value becomes erratic — concrete motivation for IAMB/HITON.


In [6]:
r = np.random.default_rng(0)
n = 4000
A0 = r.integers(0,2,n); D0 = r.integers(0,2,n)     # independent
print(f"{'|Z|':>4} | {'strata(max)':>11} | p-value")
for k in range(0, 7):
    Z = [r.integers(0,2,n) for _ in range(k)]
    p = g_test(A0, D0, Z)
    print(f"{k:>4} | {2**k:>11} | {p:.3f}")

 |Z| | strata(max) | p-value
   0 |           1 | 0.125
   1 |           2 | 0.296
   2 |           4 | 0.311
   3 |           8 | 0.940
   4 |          16 | 0.656
   5 |          32 | 0.482
   6 |          64 | 0.052


### Takeaways

- With known ground truth you can measure precision/recall/F1 directly — always
  do this before trusting a method on real data.
- Quality improves with N (power) and is tuned by α (precision/recall trade-off).
- Exploding dof from large conditioning sets is the practical reason to prefer
  algorithms that keep `|Z|` small.

**You've completed the study roadmap.** Next, move to the separate *library*
repository, where these same ideas are packaged as a tested, reusable tool — and
try the implementation exercises in `exercises/05_discovery_algorithms.md` and
`exercises/06_ci_tests.md`.
